In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import logging
import time

In [3]:
logging.basicConfig(
    level=logging.DEBUG,
    filename="parsing_log.log",
    filemode="w",
    format="%(asctime)s %(levelname)s %(message)s"
)

logging.debug("Debug информация")
logging.info("Информационное сообщение")
logging.warning("Предупреждение!")
logging.error("Ошибка")
logging.critical("Критическая ошибка!")

print("Запись в py_log.log завершена.")

Запись в py_log.log завершена.


In [4]:
url = "https://vk.com/video/trends"
page = requests.get(url)
page

<Response [200]>

In [5]:
chromedriver_path = r"/Users/aazaides/Documents/chromedriver-mac-arm64/chromedriver"
yandex_browser_binary = r"/Applications/Yandex.app/Contents/MacOS/Yandex"

options = webdriver.ChromeOptions()
options.binary_location = yandex_browser_binary

service = Service(executable_path=chromedriver_path)
driver = webdriver.Chrome(service=service, options=options)

video_data = []

try:
    driver.get(url)

    wait = WebDriverWait(driver, 20)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "VideoCard__info")))
    time.sleep(5)

    videos = driver.find_elements(By.CLASS_NAME, "VideoCard__info")
    logging.info(f"Найдено видео: {len(videos)}")

    for video in videos:
        try:
            title = video.find_element(By.CLASS_NAME, "VideoCard__title").text
            owner = video.find_element(By.CLASS_NAME, "VideoCard__ownerLink").text
            views = video.find_element(By.CLASS_NAME, "VideoCard__extendedInfoView").text
            
            verified = video.find_elements(By.CLASS_NAME, "page_verified")
            flg_verified = 1 if verified else 0
            
            video_data.append({
                "title": title,
                "owner": owner,
                "views": views,
                "flg_verified": flg_verified
            })
            
        except Exception as e:
            logging.error(f"Ошибка при обработке одного из видео: {e}")

except Exception as main_exc:
    logging.error(f"Ошибка на этапе загрузки страницы или поиска элементов: {main_exc}")

finally:
    driver.quit()

df = pd.DataFrame(video_data)
print(df.head())

                                               title                 owner  \
0                         «Маска». 6 сезон. 4 выпуск                 Маска   
1      Унижение в Белом Доме — Конец Зеленского №136  Стас Ай, Как Просто!   
2  НАРИСУЙ ЛОГОТИП ПО ПАМЯТИ ЧЕЛЛЕНДЖ или ЗАДОНАТ...                 ГЛЕНТ   
3  Lada Iskra будет хуже Гранты | Ауди закрывет з...          Асафьев Стас   
4                ФРЕНСЛЕНДЕРБОУ ► Slender Threads #1       Kuplinov ► Play   

                views  flg_verified  
0  6,6 млн просмотров             1  
1  1,1 млн просмотров             1  
2  750 тыс просмотров             1  
3  434 тыс просмотров             1  
4  376 тыс просмотров             1  


In [6]:
df.to_csv('videos_trend_pars.csv', index=False)